# 06 · Build Core

**What this does:** turns the newest raw revenue and media into the clean daily tables the MMM uses, in the BigQuery dataset `us_hment_mmm_core`. Each step checks its totals against the raw data.

| | |
|---|---|
| **When to run** | After new revenue (01) or new media (02) is loaded, and after saving title changes in 05_Review_Titles |
| **How long** | About 1–2 minutes |
| **What it changes** | Only when `DRY_RUN = False`: rebuilds `revenue_film_daily` (with the view `revenue_premium_daily`), `media_prisma_daily` and `media_tv_daily`. Raw data and film details are never changed. |
| **Safe to re-run?** | Yes. Each run rebuilds the tables from scratch from the newest data. |

### How to use it
1. Leave `DRY_RUN = True` and click **Run ▸ Run All Cells**. Read the totals in Steps 2 to 4. Nothing is written.
2. If they look right, set `DRY_RUN = False` and **Run All** again. Each step should say **✅ SAVED and checked**.
3. Set `DRY_RUN = True` again afterwards.

### What the steps use
| Step | Table | Rules |
|---|---|---|
| 2 · Revenue | `revenue_film_daily`, view `revenue_premium_daily` | US royalty only; every retailer data source (Amazon comes from "Amazon Sale" or "Amazon Royalty", which take turns; the step stops if both carry the same day); linked to films by revenue product ID. The view is the premium window (PVOD up to the day before EST) with pre-sales added to the PVOD day. |
| 3 · Media (Prisma) | `media_prisma_daily` | Advertisers ZUE, ZFF, ZFE; approved campaign budgets only; no deleted, Fee or test buys. Each buy month is spread evenly over the days of the placement's flight in that month. Linked to films by product code; non-film campaigns are kept but not added to any film. ZFF and ZFE are Focus theatrical: kept here, but the MMM uses Home Entertainment media only (ZUE, ZUV). |
| 4 · Media (TV, audio and MVPD) | `media_tv_daily` | Mediaocean NetPak (national TV) and SpotPak (local TV and radio); clients ZUE, ZUV, ZFF, ZFE (ZFF and ZFE are Focus theatrical: kept here, not used by the MMM). NetPak: air date and net cost. SpotPak: the date the station says the spot aired (otherwise the scheduled date) and 0.85 × gross cost. Linked to films by product code. |

Channel labels are added in 07_Review_Channels; the view `media_daily` combines Prisma and TV with the current labels and marks who bought the media (`media_owner`).

## Step 1 · Settings

In [ ]:
# True  = check only, nothing is written (always start here)
# False = rebuild the core tables
DRY_RUN = True

## Step 2 · Revenue

In [ ]:
from IPython.display import Markdown, display

from us_hment_mmm.core_display import show_result
from us_hment_mmm.revenue.build_core import run as build_revenue

try:
    revenue = build_revenue(dry_run=DRY_RUN)
except Exception as exc:
    display(Markdown(f"### 🛑 STOP — core revenue could not be built\n\n`{exc}`\n\nNothing was "
                     "changed by this step. Save this notebook and send it to the pipeline owner."))
    raise

show_result(revenue)

## Step 3 · Media (Prisma)

In [ ]:
from IPython.display import Markdown, display

from us_hment_mmm.core_display import show_result
from us_hment_mmm.media.build_core import run as build_media

try:
    media = build_media(dry_run=DRY_RUN)
except Exception as exc:
    display(Markdown(f"### 🛑 STOP — core media (Prisma) could not be built\n\n`{exc}`\n\nNothing was "
                     "changed by this step. Save this notebook and send it to the pipeline owner."))
    raise

show_result(media)

## Step 4 · Media (TV, audio and MVPD)

In [ ]:
from IPython.display import Markdown, display

from us_hment_mmm.core_display import show_result
from us_hment_mmm.media.build_tv import run as build_tv

try:
    tv = build_tv(dry_run=DRY_RUN)
except Exception as exc:
    display(Markdown(f"### 🛑 STOP — core media (TV, audio and MVPD) could not be built\n\n`{exc}`\n\nNothing was "
                     "changed by this step. Save this notebook and send it to the pipeline owner."))
    raise

show_result(tv)

---
**Notes on the numbers**
- Revenue films with no PVOD date are kept in `revenue_film_daily` but are not in the premium view until a date is added in 05_Review_Titles.
- Media spend dated after the snapshot is booked but not yet run. It is kept with `spend_status = 'scheduled'`; the model uses only `'actual'` spend.
- Media spend whose product code is not mapped yet is kept with link `unmapped`; map it in 05_Review_Titles and run this notebook again.